# 01 - Cohort creation

Identifies new DOAC users (2008-2022), builds continuous DOAC treatment episodes (30-day permissible gap), identifies primary-care bleeding events and serious bleeds (bleed followed by hospital admission or death within 7 days), flags warfarin use, applies the eligibility criteria and defines follow-up for the serious, minor and all-bleeding outcome cohorts.

**Manuscript items:** Study population (Methods); Fig S1 (cohort counts)

**Inputs:** CPRD Aurum patient file, DrugIssue and Observation extracts, product dictionary, dosage lookup, bleeding and hospitalisation code lists, duplicate-practice list

**Outputs:** `DERIVED_DIR/cohort_DDI_{major,minor,all}.xlsx`, `DERIVED_DIR/{major,minor,all}_events_DDI.xlsx` (patient-level - never share)

*Derived from the original analysis notebook `1_cohort_creation_DDI.ipynb`. Outputs have been cleared; no patient-level data are included.*

In [ ]:
# Project file locations are defined once in paths.py (same folder as this notebook).
# Edit DATA_DIR / OUTPUT_DIR there, or set the DDI_DATA_DIR / DDI_OUTPUT_DIR environment variables.
import os
from paths import *

In [ ]:
import time
import matplotlib.pyplot as plt
import os
import glob
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import multiprocessing as mp
from multiprocessing import Pool
from lifelines import CoxTimeVaryingFitter
import zipfile


In [ ]:
# Practices flagged by CPRD as duplicates - excluded from the cohort.
# The practice identifiers are read from DUPLICATE_PRACTICES_FILE (one ID per line).
# This file is not distributed with the repository (see data/README.md).
with open(DUPLICATE_PRACTICES_FILE) as fh:
    DUPLICATE_PRACTICE_IDS = [ln.strip() for ln in fh if ln.strip()]


In [ ]:
pats_data = pd.read_csv(PATIENT_FILE, sep=",", dtype=str)
pats_data = pats_data[pats_data["acceptable"] == "1"]
pats_data = pats_data[~pats_data["pracid"].isin(DUPLICATE_PRACTICE_IDS)]
pats_data["yob"] = pats_data["yob"].astype(int)
pats_data["regstartdate"] = pd.to_datetime(pats_data["regstartdate"], format="%d/%m/%Y",  errors="coerce")
pats_data["regenddate"] = pd.to_datetime(pats_data["regenddate"], format="%d/%m/%Y",  errors="coerce")
pats_data["cprd_ddate"] = pd.to_datetime(pats_data["cprd_ddate"], format="%d/%m/%Y",  errors="coerce")
pats_data = pats_data[["patid", "gender", "yob", "regstartdate", "regenddate", "cprd_ddate"]]
pats_data

In [ ]:
prod_dict = pd.read_csv(PRODUCT_DICT, sep="\t", dtype=str)

In [ ]:
doac_drug_names = ["apixaban", "dabigatran", "edoxaban", "rivaroxaban"]
doac_drug_codes = prod_dict[prod_dict["DrugSubstanceName"].str.contains("|".join(doac_drug_names), case=False, na=False)]


In [ ]:
dosage = pd.read_csv(os.path.join(LOOKUP_DIR, "common_dosages.txt"), sep="\t",dtype=str)


In [ ]:
def calculate_enddate(df, dosage):
    df = df.copy()
    df["daily_dose"] = df["dosageid"].map(dosage.set_index("dosageid")["daily_dose"])
    df["issuedate"] = pd.to_datetime(df["issuedate"], errors='coerce')
    df["duration"] = pd.to_numeric(df["duration"], errors="coerce")
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["daily_dose"] = pd.to_numeric(df["daily_dose"], errors="coerce").fillna(0)

    # duration==0 rows that CAN be resolved from quantity / daily dose
    mask1 = (
        (df["duration"] == 0)
        & df["quantity"].notna()
        & (df["daily_dose"] != 0)
        & (df["quantity"] != 0)
    )
    # All other duration==0 rows (daily_dose unknown/zero, OR quantity missing/zero)
    # are imputed from the per-product median below, rather than being silently
    # left at 0 (which would otherwise produce a same-day enddate for an
    # unknown/unresolvable duration).
    mask2 = (df["duration"] == 0) & (~mask1)

    df["new_duration"] = np.where(mask1, df["quantity"] / df["daily_dose"], df["duration"])
    df["new_duration"] = pd.to_numeric(df["new_duration"], errors="coerce")

    # Median computed only from rows with a genuinely recorded (non-zero) duration,
    # so neither mask1-resolved rows nor other zero-duration rows distort it.
    median_durations = (
        df.loc[df["duration"] != 0].groupby("prodcodeid")["duration"].median()
    )

    df["new_duration"] = np.where(mask2, df["prodcodeid"].map(median_durations), df["new_duration"])

    df["enddate"] = (pd.to_datetime(df["issuedate"]) + pd.to_timedelta(df["new_duration"], unit="D"))

    return df

In [ ]:
zip_path = DRUG_ISSUE_ZIP

def process_file(file_name):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep='\t', dtype=str)

    df = df[(df['prodcodeid'].isin(doac_drug_codes["ProdCodeId"])) & (df["patid"].isin(pats_data["patid"])) ]
    df["issuedate"] = pd.to_datetime(df["issuedate"], errors='coerce')
    df = df[(df["issuedate"]>= pd.to_datetime("2008-01-01") )& (df["issuedate"] <= pd.to_datetime("2022-12-31"))] 

    df = calculate_enddate(df, dosage)
    return df


if __name__ == "__main__":
    with zipfile.ZipFile(zip_path) as z:
        txt_files = [f for f in z.namelist() if f.endswith('.txt')]

    # Run in parallel
    with Pool(60) as pool:
        all_results = pool.map(process_file, txt_files)

    doac_prescriptions = pd.concat(all_results, ignore_index=True)

In [ ]:
doac_prescriptions = doac_prescriptions[["patid","issuedate", "enddate"]].drop_duplicates()
doac_prescriptions['issuedate'] = pd.to_datetime(doac_prescriptions['issuedate'])
doac_prescriptions['enddate'] = pd.to_datetime(doac_prescriptions['enddate'])
doac_prescriptions = doac_prescriptions.sort_values(['patid', 'issuedate'])


In [ ]:
def build_continuous_doac_exposure(group):
    group = group.sort_values('issuedate')

    exposures = []

    current_start = group.iloc[0]['issuedate']
    current_end = group.iloc[0]['enddate']
    patid = group.iloc[0]['patid']

    for i in range(1, len(group)):
        row = group.iloc[i]

        gap = (row['issuedate'] - current_end).days

        if gap <= 30:
            current_end = max(current_end, row['enddate'])
        else:
            # Save previous exposure
            exposures.append({
                'patid': patid,
                'study_start_date': current_start,
                'doac_end_date': current_end
            })

            # Start new exposure
            current_start = row['issuedate']
            current_end = row['enddate']

    # Append final exposure
    exposures.append({
        'patid': patid,
        'study_start_date': current_start,
        'doac_end_date': current_end
    })
    return pd.DataFrame(exposures)

In [ ]:
groups = [group for _, group in doac_prescriptions.groupby('patid')]
with mp.Pool(processes=60) as pool:
    
    results = pool.map(build_continuous_doac_exposure, groups)
doac_exposure = pd.concat(results, ignore_index=True)

In [ ]:
doac_exposure.sort_values(['patid', 'study_start_date'], inplace=True)
doac_exposure.drop_duplicates(subset="patid", keep="first", inplace=True)
doac_exposure["doac_end_date"] = doac_exposure["doac_end_date"].clip(upper=pd.to_datetime("2022-12-31"))
doac_exposure.reset_index(drop=True, inplace=True)
doac_exposure

We followed patients until a bleeding event (hosp+death), emigration, death, discontinuation of oral anticoagulation, or end of study period (31 December
2022), whichever occurred first.

In [ ]:
bleeding_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "bleeding_codes.xlsx"), dtype=str)
hospitalization_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "hospitalisation_codes.xlsx") ,dtype=str)

batches_to_consider = []
with zipfile.ZipFile(OBSERVATION_ZIP, 'r') as zip_ref:
    for batch in zip_ref.namelist():
        batches_to_consider.append(batch)

def process_file(file_name):
    
    with zipfile.ZipFile(OBSERVATION_ZIP, 'r') as zip_ref:
        df = pd.read_csv(zip_ref.open(file_name), sep="\t", dtype=str)
    
    df = df[df["patid"].isin(doac_exposure["patid"])]
    df["obsdate"] = pd.to_datetime(df["obsdate"])
    df["study_start_date"] = df["patid"].map(doac_exposure.set_index("patid")["study_start_date"])
    df["doac_end_date"] = df["patid"].map(doac_exposure.set_index("patid")["doac_end_date"])
    df = df[(df["obsdate"] > df["study_start_date"]) & (df["obsdate"] <= df["doac_end_date"])]
    bleeding_events = df[df["medcodeid"].isin(bleeding_codes["MedCodeId"])][["patid", "obsdate", "medcodeid"]]
    hospital_events = df[df["medcodeid"].isin(hospitalization_codes["MedCodeId"])][["patid", "obsdate", "medcodeid"]]

    return bleeding_events, hospital_events

if __name__ == "__main__":
    # Use a reasonable number of processes
    n_processes = 70

    with Pool(processes=n_processes) as pool:
        results = pool.map(process_file, batches_to_consider)
    bleeding_events = pd.concat([res[0] for res in results], ignore_index=True)
    hospital_events = pd.concat([res[1] for res in results], ignore_index=True)

In [ ]:
bleeding_events = bleeding_events.drop_duplicates(subset=["patid", "obsdate"], keep="first")
hospital_events = hospital_events.drop_duplicates(subset=["patid", "obsdate"], keep="first")

In [ ]:
pats_data["study_start_date"] = pats_data["patid"].map(doac_exposure.set_index("patid")["study_start_date"])
pats_data["doac_end_date"] = pats_data["patid"].map(doac_exposure.set_index("patid")["doac_end_date"])

pats_data["study_start_date"] = pd.to_datetime(pats_data["study_start_date"], errors="coerce")
pats_data = pats_data[pats_data["patid"].isin(doac_exposure["patid"])]
pats_data["cprd_ddate"] = pd.to_datetime(pats_data["cprd_ddate"])
pats_data

In [ ]:
death_events = pats_data[(pats_data["cprd_ddate"]>pats_data["study_start_date"]) & (pats_data["cprd_ddate"]<=pats_data["doac_end_date"])]
death_events = death_events[["patid", "cprd_ddate"]]
death_events.rename(columns={"cprd_ddate":"obsdate"}, inplace=True)


hospital_events = hospital_events[["patid","obsdate"]]
hospital_events = pd.concat([hospital_events, death_events], axis=0)
hospital_events.drop_duplicates(inplace=True)
hospital_events.sort_values(["patid", "obsdate"], inplace=True)
hospital_events.reset_index(drop=True, inplace=True)

In [ ]:
# merge bleeding events with hospitalization events
bleeding_hospitalizations = pd.merge(hospital_events, bleeding_events, on="patid", how="inner", suffixes=('_hospital', '_bleeding'))
bleeding_hospitalizations = bleeding_hospitalizations[(bleeding_hospitalizations["obsdate_hospital"]>=bleeding_hospitalizations["obsdate_bleeding"]) &
                        (bleeding_hospitalizations["obsdate_hospital"]<= bleeding_hospitalizations["obsdate_bleeding"] + pd.Timedelta(days=7))]
bleeding_hospitalizations.sort_values(by=["patid", "obsdate_bleeding", "obsdate_hospital"], inplace=True)
#select first hospitalization within 7 days of bleeding event
bleeding_hospitalizations = bleeding_hospitalizations.groupby(["patid", "obsdate_bleeding"]).first().reset_index()
bleeding_hospitalizations.sort_values(by=["patid", "obsdate_bleeding"], inplace=True)
bleeding_hospitalizations.drop_duplicates(subset=["patid"], inplace=True)
#bleeding_hospitalizations["term"] = bleeding_hospitalizations["medcodeid"].map(med_codes.set_index("MedCodeId")["Term"])
major_events_df = bleeding_hospitalizations.copy()
major_events_df.rename(columns={"obsdate_bleeding":"obsdate"}, inplace=True)
major_events_df

In [ ]:
major_keys = set(zip(bleeding_hospitalizations["patid"], bleeding_hospitalizations["obsdate_bleeding"]))

bleeding_events["is_major"] = (list(zip(bleeding_events["patid"], bleeding_events["obsdate"])))
bleeding_events["is_major"] = bleeding_events["is_major"].isin(major_keys).astype(int)

In [ ]:
warfarin_codes = prod_dict[prod_dict["DrugSubstanceName"].str.contains("warfarin", case=False, na=False)]

zip_path = DRUG_ISSUE_ZIP

def process_file(file_name):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep='\t', dtype=str)

    # Filtering
    df = df[(df['prodcodeid'].isin(warfarin_codes["ProdCodeId"])) & (df['patid'].isin(doac_exposure["patid"].tolist()))]
    df["study_start_date"] = df["patid"].map(doac_exposure.set_index("patid")["study_start_date"])
    df["doac_end_date"] = df["patid"].map(doac_exposure.set_index("patid")["doac_end_date"])

    df = calculate_enddate(df, dosage)
    # ── Window definitions ──────────────────────────────────────────────────────
    df['window_start_1w'] = df['study_start_date'] - pd.Timedelta(days=7)
    df['window_start_1m'] = df['study_start_date'] - pd.Timedelta(days=30)

    # Warfarin issued BEFORE study_start_date and prescription end date falls
    # within the look-back window  (issue ended no earlier than window start)
    df['warfarin_in_1w_window'] = (
        (df['issuedate'] <= df['study_start_date']) &
        (df['enddate'] >= df['window_start_1w'])
    )
    df['warfarin_in_1m_window'] = (
        (df['issuedate'] <= df['study_start_date']) &
        (df['enddate'] >= df['window_start_1m'])
    )

    taken_warfarin_1w = df[df['warfarin_in_1w_window']]['patid'].unique().tolist()
    taken_warfarin_1m = df[df['warfarin_in_1m_window']]['patid'].unique().tolist()
    taken_warfarin_during_study = df[(df['issuedate'] >= df['study_start_date']) & (df['issuedate'] <= df['doac_end_date'])]['patid'].unique().tolist()

    return taken_warfarin_1w, taken_warfarin_1m, taken_warfarin_during_study


if __name__ == "__main__":
    with zipfile.ZipFile(zip_path) as z:
        txt_files = [f for f in z.namelist() if f.endswith('.txt')]

    # Run in parallel
    with Pool(60) as pool:
        all_results = pool.map(process_file, txt_files)

    # Unzip the two lists from each file's result tuple
    results_1w, results_1m, results_during_study = zip(*all_results)

    pats_warfarin_1w_before_doac = set(patid for sublist in results_1w for patid in sublist)
    pats_warfarin_1m_before_doac = set(patid for sublist in results_1m for patid in sublist)
    pats_warfarin_during_study = set(patid for sublist in results_during_study for patid in sublist)

In [ ]:
print(f"Patients with warfarin in 1 week window: {len(pats_warfarin_1w_before_doac)}")
print(f"Patients with warfarin in 1 month window: {len(pats_warfarin_1m_before_doac)}")
print(f"Patients with warfarin during study: {len(pats_warfarin_during_study)}")

doac_exposure["taken_warfarin_1w_before"] = doac_exposure["patid"].isin(pats_warfarin_1w_before_doac).astype(int)
doac_exposure["taken_warfarin_1m_before"] = doac_exposure["patid"].isin(pats_warfarin_1m_before_doac).astype(int)
doac_exposure["taken_warfarin_during_study"] = doac_exposure["patid"].isin(pats_warfarin_during_study).astype(int)

In [ ]:
cohort_major = doac_exposure.copy()
cohort_major["age_at_study_start"] = cohort_major["study_start_date"].dt.year - cohort_major["patid"].map(pats_data.set_index("patid")["yob"])
cohort_major["event_date"] = cohort_major["patid"].map(major_events_df.set_index("patid")["obsdate"])
cohort_major["event_date"] = pd.to_datetime(cohort_major["event_date"], errors="coerce")
cohort_major["regstartdate"] = cohort_major["patid"].map(pats_data.set_index("patid")["regstartdate"])
cohort_major["regenddate"] = cohort_major["patid"].map(pats_data.set_index("patid")["regenddate"])
cohort_major = cohort_major[~(cohort_major["regenddate"]<cohort_major["study_start_date"])]
cohort_major["atleast_1y_med_hist_flag"] = cohort_major["regstartdate"] <= (cohort_major["study_start_date"] - pd.Timedelta(days=365))
cohort_major["atleast_1y_med_hist_flag"] = cohort_major["atleast_1y_med_hist_flag"].astype(int)

bleeding_date     = cohort_major["event_date"]
registration_date = cohort_major["regenddate"]
doac_end_date     = cohort_major["doac_end_date"]
followup_end_date = pd.Series(pd.to_datetime("2022-12-31"), index=cohort_major.index)

# All candidate dates that could end follow-up
candidates = pd.DataFrame({
    "bleeding_event":   bleeding_date,
    "registration_end": registration_date,
    "doac_end":         doac_end_date,
    "follow_up_end":    followup_end_date,
})

# True study end = earliest non-null candidate date.
# Column order only matters as a tie-break (bleeding_event wins ties, same as original <= vs < intent).
cohort_major["study_end_date"] = candidates.min(axis=1, skipna=True)
cohort_major["reason_for_end"] = candidates.idxmin(axis=1, skipna=True)

cohort_major["major_event_flag"] = np.where(cohort_major["reason_for_end"]!="bleeding_event", 0, 1)
cohort_major["event_date"] = np.where(cohort_major["reason_for_end"]!="bleeding_event", pd.NaT, cohort_major["event_date"])
cohort_major["reason_for_end"] = np.where((cohort_major["study_end_date"]==pd.to_datetime("2022-12-31")) & (cohort_major["reason_for_end"]!="bleeding_event") ,
                                            "follow_up_end", cohort_major["reason_for_end"])

cohort_major["event_date"] = pd.to_datetime(cohort_major["event_date"], errors="coerce")
# Drop the intermediate column; keep event_date for the analysis.
cohort_major = cohort_major.drop(columns=["regstartdate", "regenddate", "doac_end_date"])

In [ ]:
pats_of_our_cohort = cohort_major[(cohort_major["age_at_study_start"] >= 18) &
                (cohort_major["atleast_1y_med_hist_flag"] == 1) &
                (cohort_major["taken_warfarin_1w_before"] ==0) &
                 (cohort_major["taken_warfarin_during_study"] ==0) ]["patid"].unique().tolist()

print(f"Number of patients in the cohort: {len(pats_of_our_cohort)}")

In [ ]:
cohort_major[(cohort_major["patid"].isin(pats_of_our_cohort)) & 
              (cohort_major["reason_for_end"]=="bleeding_event") &
              (cohort_major["major_event_flag"]==1) ]["patid"].nunique()

In [ ]:
cohort_major[(cohort_major["patid"].isin(pats_of_our_cohort))]["reason_for_end"].value_counts()

In [ ]:
major_events_df = major_events_df[major_events_df["patid"].isin(cohort_major[cohort_major["event_date"].notnull()]["patid"].unique())]
major_events_df

In [ ]:
cohort_major.to_excel(os.path.join(DERIVED_DIR, "cohort_DDI_major.xlsx"), index=False)
major_events_df.to_excel(os.path.join(DERIVED_DIR, "major_events_DDI.xlsx"), index=False)

In [ ]:
all_events_df = bleeding_events.sort_values(["patid", "obsdate"])
all_events_df.drop_duplicates(subset=["patid"], keep="first", inplace=True)

In [ ]:
cohort_all = doac_exposure.copy()
cohort_all["age_at_study_start"] = cohort_all["study_start_date"].dt.year - cohort_all["patid"].map(pats_data.set_index("patid")["yob"])
cohort_all["event_date"] = cohort_all["patid"].map(all_events_df.set_index("patid")["obsdate"])
cohort_all["event_date"] = pd.to_datetime(cohort_all["event_date"], errors="coerce")
cohort_all["regstartdate"] = cohort_all["patid"].map(pats_data.set_index("patid")["regstartdate"])
cohort_all["regenddate"] = cohort_all["patid"].map(pats_data.set_index("patid")["regenddate"])
cohort_all = cohort_all[~(cohort_all["regenddate"]<cohort_all["study_start_date"])]
cohort_all["atleast_1y_med_hist_flag"] = cohort_all["regstartdate"] <= (cohort_all["study_start_date"] - pd.Timedelta(days=365))
cohort_all["atleast_1y_med_hist_flag"] = cohort_all["atleast_1y_med_hist_flag"].astype(int)

bleeding_date     = cohort_all["event_date"]
registration_date = cohort_all["regenddate"]
doac_end_date     = cohort_all["doac_end_date"]
followup_end_date = pd.Series(pd.to_datetime("2022-12-31"), index=cohort_all.index)

# All candidate dates that could end follow-up
candidates = pd.DataFrame({
    "bleeding_event":   bleeding_date,
    "registration_end": registration_date,
    "doac_end":         doac_end_date,
    "follow_up_end":    followup_end_date,
})

# True study end = earliest non-null candidate date.
# Column order only matters as a tie-break (bleeding_event wins ties, same as original <= vs < intent).
cohort_all["study_end_date"] = candidates.min(axis=1, skipna=True)
cohort_all["reason_for_end"] = candidates.idxmin(axis=1, skipna=True)

cohort_all["major_event_flag"] = np.where(cohort_all["reason_for_end"]!="bleeding_event", 0, 1)
cohort_all["event_date"] = np.where(cohort_all["reason_for_end"]!="bleeding_event", pd.NaT, cohort_all["event_date"])
cohort_all["reason_for_end"] = np.where((cohort_all["study_end_date"]==pd.to_datetime("2022-12-31")) & (cohort_all["reason_for_end"]!="bleeding_event") ,
                                            "follow_up_end", cohort_all["reason_for_end"])

cohort_all["event_date"] = pd.to_datetime(cohort_all["event_date"], errors="coerce")
# Drop the intermediate column; keep event_date for the analysis.
cohort_all = cohort_all.drop(columns=["regstartdate", "regenddate", "doac_end_date"])
cohort_all["event_date"] = pd.to_datetime(cohort_all["event_date"], errors="coerce")

In [ ]:
cohort_all.to_excel(os.path.join(DERIVED_DIR, "cohort_DDI_all.xlsx"), index=False)
all_events_df.to_excel(os.path.join(DERIVED_DIR, "all_events_DDI.xlsx"), index=False)

In [ ]:
minor_events_df = bleeding_events[bleeding_events["is_major"] == 0].sort_values(["patid", "obsdate"])
minor_events_df.drop_duplicates(subset=["patid"], keep="first", inplace=True)

In [ ]:
cohort_minor = doac_exposure.copy()
cohort_minor["age_at_study_start"] = cohort_minor["study_start_date"].dt.year - cohort_minor["patid"].map(pats_data.set_index("patid")["yob"])
cohort_minor["event_date"] = cohort_minor["patid"].map(minor_events_df.set_index("patid")["obsdate"])
cohort_minor["event_date"] = pd.to_datetime(cohort_minor["event_date"], errors="coerce")
cohort_minor["regstartdate"] = cohort_minor["patid"].map(pats_data.set_index("patid")["regstartdate"])
cohort_minor["regenddate"] = cohort_minor["patid"].map(pats_data.set_index("patid")["regenddate"])
cohort_minor = cohort_minor[~(cohort_minor["regenddate"]<cohort_minor["study_start_date"])]
cohort_minor["atleast_1y_med_hist_flag"] = cohort_minor["regstartdate"] <= (cohort_minor["study_start_date"] - pd.Timedelta(days=365))
cohort_minor["atleast_1y_med_hist_flag"] = cohort_minor["atleast_1y_med_hist_flag"].astype(int)

bleeding_date     = cohort_minor["event_date"]
registration_date = cohort_minor["regenddate"]
doac_end_date     = cohort_minor["doac_end_date"]
followup_end_date = pd.Series(pd.to_datetime("2022-12-31"), index=cohort_minor.index)

# All candidate dates that could end follow-up
candidates = pd.DataFrame({
    "bleeding_event":   bleeding_date,
    "registration_end": registration_date,
    "doac_end":         doac_end_date,
    "follow_up_end":    followup_end_date,
})

# True study end = earliest non-null candidate date.
# Column order only matters as a tie-break (bleeding_event wins ties, same as original <= vs < intent).
cohort_minor["study_end_date"] = candidates.min(axis=1, skipna=True)
cohort_minor["reason_for_end"] = candidates.idxmin(axis=1, skipna=True)

cohort_minor["major_event_flag"] = np.where(cohort_minor["reason_for_end"]!="bleeding_event", 0, 1)
cohort_minor["event_date"] = np.where(cohort_minor["reason_for_end"]!="bleeding_event", pd.NaT, cohort_minor["event_date"])
cohort_minor["reason_for_end"] = np.where((cohort_minor["study_end_date"]==pd.to_datetime("2022-12-31")) & (cohort_minor["reason_for_end"]!="bleeding_event") ,
                                            "follow_up_end", cohort_minor["reason_for_end"])

cohort_minor["event_date"] = pd.to_datetime(cohort_minor["event_date"], errors="coerce")
# Drop the intermediate column; keep event_date for the analysis.
cohort_minor = cohort_minor.drop(columns=["regstartdate", "regenddate", "doac_end_date"])

In [ ]:
cohort_minor[(cohort_minor["patid"].isin(pats_of_our_cohort)) & 
              (cohort_minor["reason_for_end"]=="bleeding_event") &
              (cohort_minor["major_event_flag"]==1) ]["patid"].nunique()

In [ ]:
cohort_minor[(cohort_minor["patid"].isin(pats_of_our_cohort))]["reason_for_end"].value_counts()

In [ ]:
cohort_minor["event_date"] = pd.to_datetime(cohort_minor["event_date"], errors="coerce")

In [ ]:
minor_events_df = minor_events_df[minor_events_df["patid"].isin(cohort_minor[cohort_minor["event_date"].notnull()]["patid"].unique())]
minor_events_df.drop(columns=["is_major"], inplace=True)
minor_events_df

In [ ]:
cohort_minor.to_excel(os.path.join(DERIVED_DIR, "cohort_DDI_minor.xlsx"), index=False)
minor_events_df.to_excel(os.path.join(DERIVED_DIR, "minor_events_DDI.xlsx"), index=False)